# PCA-50 Results Overview

This notebook only analyzes eval-adoption probe runs whose result-group name contains `pca50`.

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

pd.options.display.max_columns = 200
pd.options.display.max_rows = 200

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent

RESULTS_DIR = ROOT / "results"
print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")

In [ ]:
EVAL_ADOPTION_PROBE_RE = re.compile(
    r"^(?:eval-adoption-)?absolute_accuracy_decay__(?P<perturbation_type>.+?)__(?:(?P<probe_control>none|randomization|permutation)__)?L(?P<layer>\d{3})$"
)

def parse_run_path(path: Path) -> dict:
    rel = path.relative_to(RESULTS_DIR)
    parts = rel.parts
    if len(parts) < 8:
        raise ValueError(f"Unexpected results path: {rel}")

    probe_idx = next(
        (i for i, part in enumerate(parts) if EVAL_ADOPTION_PROBE_RE.match(part) or re.search(r"(?:_L|__L)\d{3}$", part)),
        None,
    )
    if probe_idx is None:
        raise ValueError(f"Could not locate probe segment in {rel}")

    probe_name = parts[probe_idx]
    tail = parts[probe_idx + 1 :]
    if len(tail) == 7:
        model_name, encoding, control_task, sample_size, seed, fold, filename = tail
        status = "completed"
        version = ""
    elif len(tail) == 8:
        model_name, encoding, control_task, sample_size, seed, fold, status, filename = tail
        version = ""
    elif len(tail) == 9:
        model_name, encoding, control_task, sample_size, seed, fold, status, version, filename = tail
    else:
        raise ValueError(f"Unexpected run tail after probe {probe_name}: {tail}")

    match = EVAL_ADOPTION_PROBE_RE.match(probe_name)
    layer = int(match.group("layer")) if match else int(re.search(r"(?:_L|__L)(\d{3})", probe_name).group(1))
    perturbation_type = match.group("perturbation_type") if match else None
    probe_control = match.group("probe_control") if match else None

    return {
        "results_group": "/".join(parts[:probe_idx]),
        "probe_name": probe_name,
        "layer": layer,
        "model_name": model_name,
        "encoding": encoding,
        "control_task": control_task,
        "probe_control": probe_control or control_task.lower(),
        "perturbation_type": perturbation_type,
        "sample_size": int(sample_size),
        "fold": int(fold),
        "seed": int(seed),
        "status": status,
        "version": version,
        "filename": filename,
        "path": path,
        "mtime": path.stat().st_mtime,
    }

def last_non_null(series: pd.Series):
    cleaned = series.dropna()
    return cleaned.iloc[-1] if not cleaned.empty else np.nan

def maybe_metric(df: pd.DataFrame, column: str, reducer: str = "last"):
    if column not in df.columns:
        return np.nan
    series = df[column]
    if reducer == "last":
        return last_non_null(series)
    if reducer == "max":
        return series.max(skipna=True)
    if reducer == "min":
        return series.min(skipna=True)
    raise ValueError(reducer)

def load_metrics() -> tuple[pd.DataFrame, pd.DataFrame]:
    run_rows = []
    curve_frames = []
    for metrics_path in sorted(RESULTS_DIR.rglob("metrics.csv")):
        info = parse_run_path(metrics_path)
        if "pca50" not in info["results_group"].lower():
            continue
        df = pd.read_csv(metrics_path)
        curve = df.copy()
        for key, value in info.items():
            if key not in {"path", "filename"}:
                curve[key] = value
        curve_frames.append(curve)
        run_rows.append({
            **{k: v for k, v in info.items() if k not in {"path", "filename"}},
            "metrics_path": str(metrics_path),
            "epochs_logged": len(df),
            "best_val_loss": maybe_metric(df, "val loss", "min"),
            "best_val_error": maybe_metric(df, "val error", "min"),
            "best_val_pearson": maybe_metric(df, "val pearson", "max"),
            "full_test_error": maybe_metric(df, "full test error", "last"),
            "full_test_pearson": maybe_metric(df, "full test pearson", "last"),
        })
    if not run_rows:
        raise RuntimeError(f"No pca50 eval-adoption metrics found under {RESULTS_DIR}")
    runs_df = pd.DataFrame(run_rows).sort_values(["results_group", "layer", "control_task", "perturbation_type"]).reset_index(drop=True)
    curves_df = pd.concat(curve_frames, ignore_index=True).sort_values(["results_group", "layer", "control_task", "step"]) if curve_frames else pd.DataFrame()
    return runs_df, curves_df

runs_df, curves_df = load_metrics()


In [ ]:
latest_group = runs_df.groupby("results_group")["mtime"].max().sort_values(ascending=False).index[0]
latest_runs = runs_df[runs_df["results_group"] == latest_group].copy()
latest_runs = latest_runs[latest_runs["control_task"].isin(["NONE", "RANDOMIZATION"])].copy()
latest_runs["control_label"] = latest_runs["control_task"].str.lower()

pd.DataFrame({
    "latest_results_group": [latest_group],
    "models": [", ".join(sorted(latest_runs["model_name"].unique()))],
    "controls": [", ".join(sorted(latest_runs["control_task"].unique()))],
    "perturbation_types": [", ".join(sorted(latest_runs["perturbation_type"].dropna().unique()))],
    "layers": [latest_runs["layer"].nunique()],
    "n_runs": [len(latest_runs)],
})

## Summary

In [ ]:
summary = (
    latest_runs.groupby(["control_label", "perturbation_type"])
    .agg(
        best_full_test_pearson=("full_test_pearson", "max"),
        min_full_test_error=("full_test_error", "min"),
    )
    .round(4)
    .reset_index()
    .sort_values(["control_label", "best_full_test_pearson"], ascending=[True, False])
)
summary


## Pearson Facet Plots

In [ ]:
fig = px.line(
    latest_runs.sort_values("layer"),
    x="layer",
    y="full_test_pearson",
    color="perturbation_type",
    facet_col="control_label",
    markers=True,
    category_orders={"control_label": ["none", "randomization"]},
    title="PCA-50 Full-Test Pearson by Layer, Control Task, and Perturbation Type",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("control_label=", "control=")))
fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="Pearson correlation")
fig.show()


## Error Facet Plot

In [ ]:
fig = px.line(
    latest_runs.sort_values("layer"),
    x="layer",
    y="full_test_error",
    color="perturbation_type",
    facet_col="control_label",
    markers=True,
    category_orders={"control_label": ["none", "randomization"]},
    title="PCA-50 Full-Test Error by Layer, Control Task, and Perturbation Type",
)
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("control_label=", "control=")))
fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="Full-test error")
fig.show()

## Heatmap Views

In [ ]:
fig = px.scatter(
    latest_runs,
    x="layer",
    y="perturbation_type",
    color="full_test_pearson",
    facet_col="control_label",
    category_orders={"control_label": ["none", "randomization"]},
    color_continuous_scale="RdBu",
    range_color=[-1, 1],
    title="PCA-50 Full-Test Pearson Heatmap View",
)
fig.update_traces(marker={"size": 14, "symbol": "square"})
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("control_label=", "control=")))
fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="Perturbation type")
fig.show()

In [ ]:
fig = px.scatter(
    latest_runs,
    x="layer",
    y="perturbation_type",
    color="full_test_error",
    facet_col="control_label",
    category_orders={"control_label": ["none", "randomization"]},
    color_continuous_scale="Viridis_r",
    title="PCA-50 Full-Test Error Heatmap View",
)
fig.update_traces(marker={"size": 14, "symbol": "square"})
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("control_label=", "control=")))
fig.update_xaxes(title_text="Layer")
fig.update_yaxes(title_text="Perturbation type")
fig.show()